<p><font size="6" color='grey'> <b>
KI-Agenten. Planen. Handeln. Prüfen.
</b></font> </br></p>



<p><font size="5" color='grey'> <b>
Gradio UI für Agenten
</b></font> </br></p>

---

**Beitrag zum Leitprojekt:** Die UI des Meeting- & Research-Briefing-Agenten ist kein normales Chat-Fenster — sie muss Quellen, Trace-Informationen, Sicherheitseinschätzung und Freigabe-Status sichtbar machen, nicht nur den Antworttext. Eine Oberfläche, die Kontrolle verschweigt, untergräbt die Leitplanken aus der Leitaufgabe, egal wie gut das System dahinter ist.

In [19]:
#@title 🛠️ Umgebung einrichten{ display-mode: "form" }
!uv pip install --system -q git+https://github.com/ralf-42/Agenten.git#subdirectory=04_modul

import os
os.environ["LANGSMITH_TRACING"]  = "true"
os.environ["LANGSMITH_PROJECT"]  = "M30-Gradio-UI"
os.environ["LANGSMITH_ENDPOINT"] = "https://eu.api.smith.langchain.com"

from genai_lib.utilities import (
    check_environment,
    get_ipinfo,
    setup_api_keys,
    mprint,
    install_packages,
    mermaid,
    get_model_profile,
    extract_thinking,
    load_prompt,
    show_trace
)

setup_api_keys(['OPENAI_API_KEY', 'LANGSMITH_API_KEY'], create_globals=False)
print()
check_environment()
print()
get_ipinfo()

# Modell-Konfiguration — Rollen als Konstanten
from genai_lib.model_config import BASELINE, ROUTER, JUDGE, PLANNER, WORKER, WORKER_PREMIUM, CODING, EMBEDDINGS
# LangSmith Tracing
run_cfg = {
    "run_name": "M30_Gradio_UI_fuer_Agenten",
    "tags": ["m30", "gradio"],
    "metadata": {"notebook": "M30", "version": "1.0"}
}


✓ OPENAI_API_KEY erfolgreich gesetzt
✓ LANGSMITH_API_KEY erfolgreich gesetzt

Python Version: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]

Installierte LangChain- und LangGraph-Bibliotheken:
langchain                                1.2.15
langchain-chroma                         1.1.0
langchain-classic                        1.0.4
langchain-community                      0.4.1
langchain-core                           1.3.1
langchain-ollama                         1.1.0
langchain-openai                         1.2.0
langchain-text-splitters                 1.1.2
langgraph                                1.1.9
langgraph-checkpoint                     4.0.2
langgraph-prebuilt                       1.0.10
langgraph-sdk                            0.3.13

IP-Adresse: 34.150.169.251
Hostname: 251.169.150.34.bc.googleusercontent.com
Stadt: Washington
Region: District of Columbia
Land: US
Koordinaten: 38.8951,-77.0364
Provider: AS396982 Google LLC
Postleitzahl: 20004
Zeitzone: America/New

# 1 | Übersicht
---


Gradio macht den Meeting- & Research-Briefing-Agenten interaktiv nutzbar: Fragen werden gestellt, Quellen werden sichtbar, Unsicherheit wird angezeigt und kritische Antworten können vor der Ausgabe freigegeben werden.

| Gradio-Komponente | Einsatz im Meeting-Briefing-Agenten |
|-------------------|-------------------------------|
| `gr.ChatInterface` | Schnellste Chat-UI für Fragen an den Korpus |
| `gr.Blocks` | Chat, Quellenpanel und Tool-Log in einem Layout |
| `gr.State` | Sitzungsstatus, ausstehende Freigaben, Verlauf |
| Generator / `yield` | Fortschrittsschritte beim Retrieval anzeigen |
| Buttons | Human-in-the-Loop: Freigeben / Ablehnen |

# 2 | gr.ChatInterface — Research Chat
---


`gr.ChatInterface` ist die schnellste Variante: eine Python-Funktion `fn(message, history) -> str` wird automatisch als Chat-UI gerendert.

```python
import gradio as gr

def chat_fn(message: str, history: list) -> str:
    return briefing_agent.invoke(...)

gr.ChatInterface(fn=chat_fn).launch()
```

| Parameter | Bedeutung |
|-----------|----------|
| `fn` | Callback-Funktion `(message, history) → str` |
| `title` | Überschrift der UI |
| `examples` | Klickbare Beispiel-Fragen |
| `type="messages"` | Modernes Nachrichtenformat (Gradio 4+) |
| `share=True` | Öffentliche URL für Colab/Remote |

In [ ]:
#@markdown   <p><font size="4" color='green'>  ChatInterface Architektur</font> </br></p>

diagram = """
%%{init: {'theme':'dark'}}%%
flowchart LR
    U(["Frage"])
    CI["gr.ChatInterface
chat_fn(message, history)"]
    A["Briefing-Agent
Antwort + Quellen"]
    T["Tools
suche_quellen, pruefe_quellenbindung"]

    U -->|Eingabe| CI
    CI -->|message| A
    A -->|Tool Call| T
    T -->|Quelle + Bewertung| A
    A -->|strukturierte Antwort| CI
    CI -->|Antwort mit Quellen| U

    style CI  fill:#1565C0,color:#fff
    style A   fill:#2E7D32,color:#fff
    style T   fill:#37474F,color:#fff
    style U   fill:#E65100,color:#fff
"""
mermaid(diagram, width=1100)


In [ ]:
from langchain.chat_models import init_chat_model
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage, ToolMessage
from langchain.agents import create_agent

from genai_lib.model_config import WORKER
llm = init_chat_model(WORKER)

BRIEFING_SNIPPETS = {
    "17.03": {
        "quelle": "protokoll_steuerkreis_2026-03-17.pdf",
        "passage": "Der Steuerkreis beschließt Entscheidung D1: ChromaDB wird als Vektordatenbank für den Prototyp evaluiert.",
        "sicherheit": "hoch",
    },
    "risiko": {
        "quelle": "protokoll_steuerkreis_2026-04-28.pdf",
        "passage": "Risiko R2 (Datenschutzfreigabe) ist mitigiert, die Freigabe von Dr. Brandt liegt vor.",
        "sicherheit": "hoch",
    },
    "14.04": {
        "quelle": "protokoll_steuerkreis_2026-04-14.pdf",
        "passage": "Entscheidung D2 ersetzt D1: Statt ChromaDB wird eine verwaltete Cloud-Vektordatenbank für den produktiven Einsatz beschlossen.",
        "sicherheit": "mittel",
    },
}


def _waehle_snippet(frage: str) -> dict:
    text = frage.lower()
    for key, snippet in BRIEFING_SNIPPETS.items():
        if key in text:
            return snippet
    return {
        "quelle": "Nicht im Korpus",
        "passage": "Keine passende Passage im Demo-Korpus gefunden.",
        "sicherheit": "niedrig",
    }


@tool
def suche_quellen(frage: str) -> str:
    """Sucht eine passende Demo-Quelle im Projekt-Korpus."""
    snippet = _waehle_snippet(frage)
    return f"Quelle: {snippet['quelle']}\nPassage: {snippet['passage']}\nSicherheit: {snippet['sicherheit']}"


@tool
def pruefe_quellenbindung(antwort: str) -> str:
    """Prüft, ob eine Antwort eine Quelle oder Nicht-im-Korpus-Hinweis enthält."""
    lower = antwort.lower()
    if "quelle:" in lower or "nicht im korpus" in lower:
        return "Quellenbindung: ok"
    return "Quellenbindung: fehlt"


tools_liste = [suche_quellen, pruefe_quellenbindung]
briefing_agent = create_agent(
    model=llm,
    tools=tools_liste,
    system_prompt=(
        "Rolle: Meeting- & Research-Briefing-Agent für Projekt Kompass. "
        "Antworten brauchen Quelle, Passage und Sicherheit. "
        "Wenn keine Passage vorhanden ist: Nicht im Korpus schreiben."
    ),
)
print("✅ Meeting-Briefing-Agent mit Quellen- und Prüf-Tool bereit")


In [ ]:
import gradio as gr


def chat_fn(message: str, history: list) -> str:
    """Gradio-Callback: gibt eine quellengebundene Briefing-Antwort zurück."""
    result = briefing_agent.invoke(
        {"messages": [HumanMessage(content=message, config=run_cfg)]},
        config={"run_name": "M30-BasicBriefingChat", "tags": ["m30", "basic-chat", "briefing"]},
    )
    return result["messages"][-1].content


basic_probe = chat_fn("Was wurde am 17.03.2026 zur Vektordatenbank entschieden?", [])
print(basic_probe[:500])

demo_basic = gr.ChatInterface(
    fn=chat_fn,
    title="Meeting-Briefing-Agent",
    description="Fragen an den Projekt-Korpus mit Quelle, Passage und Sicherheit.",
    examples=[
        "Was wurde am 17.03.2026 zur Vektordatenbank entschieden?",
        "Welchen Status hat Risiko R2?",
        "Was hat sich am 14.04.2026 geändert?",
    ],
    type="messages",
)
demo_basic.launch(quiet=True)


# 3 | gr.Blocks — Chat mit Quellen- und Tool-Visualisierung
---


`gr.Blocks` erlaubt ein Layout mit mehreren Panels. Hier: Chat links, Quellen und Tool-Log rechts.

```
┌─────────────────────────┬────────────────────────────┐
│  Chat                   │  Quellen / Tool-Aufrufe    │
│  Frage an den Korpus    │  suche_quellen(...)        │
│  Antwort                │  Quelle + Sicherheit       │
└─────────────────────────┴────────────────────────────┘
```


In [ ]:
import gradio as gr


def verarbeite(nachricht: str, verlauf: list):
    """Führt den Briefing-Agenten aus und extrahiert Tool-Aufrufe für den Log."""
    verlauf = verlauf or []
    result = briefing_agent.invoke(
        {"messages": [HumanMessage(content=nachricht, config=run_cfg)]},
        config={"run_name": "M30-BlocksBriefingChat", "tags": ["m30", "blocks", "briefing"]},
    )

    tool_log_lines = []
    for msg in result["messages"]:
        if hasattr(msg, "tool_calls") and msg.tool_calls:
            for tc in msg.tool_calls:
                args_str = ", ".join(f"{k}={v}" for k, v in tc["args"].items())
                tool_log_lines.append(f"▶ {tc['name']}({args_str})")
        if isinstance(msg, ToolMessage):
            tool_log_lines.append(f"↳ {msg.content}")

    antwort = result["messages"][-1].content
    verlauf.append({"role": "user", "content": nachricht})
    verlauf.append({"role": "assistant", "content": antwort})
    tool_text = "\n".join(tool_log_lines) if tool_log_lines else "(keine Tools aufgerufen)"
    return "", verlauf, tool_text


blocks_probe = verarbeite("Welchen Status hat Risiko R2?", [])
print(blocks_probe[2][:500])

with gr.Blocks(title="Meeting-Briefing-Agent mit Quellen", theme=gr.themes.Soft()) as demo_blocks:
    gr.Markdown("## Meeting-Briefing-Agent mit Quellenpanel")
    with gr.Row():
        with gr.Column(scale=3):
            chatbot = gr.Chatbot(type="messages", label="Chat", height=300)
        with gr.Column(scale=2):
            tool_box = gr.Textbox(
                label="Quellen und Tool-Log", lines=12,
                interactive=False, placeholder="Noch keine Quellen geprüft..."
            )
    with gr.Row():
        eingabe = gr.Textbox(placeholder="Frage an den Projekt-Korpus", label="Eingabe", scale=4)
        senden = gr.Button("Senden", variant="primary", scale=1)

    senden.click(verarbeite, [eingabe, chatbot], [eingabe, chatbot, tool_box])
    eingabe.submit(verarbeite, [eingabe, chatbot], [eingabe, chatbot, tool_box])

demo_blocks.launch(quiet=True)


# 4 | Streaming — Recherchefortschritt anzeigen
---


LangGraph unterstützt `stream()` — damit lassen sich Zwischenschritte an Gradio weiterleiten. Für Research-Workflows ist oft wichtiger, den Fortschritt sichtbar zu machen: Anfrage prüfen, Quellen suchen, Antwort formulieren.

> **Hinweis:** `stream()` liefert Chunks pro Node — nicht zwingend pro Token. Für echtes Token-Streaming ist `astream_events()` nötig.


In [24]:
#@markdown   <p><font size="4" color='green'>  Streaming-Flow</font> </br></p>

diagram = '''
%%{init: {'theme':'forest'}}%%
sequenceDiagram
    autonumber
    participant U as 👤 User
    participant G as gr.ChatInterface
    participant LG as LangGraph
    participant T as Tools

    U->>G: Frage
    G->>LG: stream({"messages": [...]})
    loop Chunk für Chunk
        LG-->>G: {"agent": {"messages": [...]}}
        G-->>U: yield partial_antwort
    end
    LG->>T: Tool Call
    T-->>LG: Ergebnis
    LG-->>G: {"agent": {"messages": [finale Antwort]}}
    G-->>U: yield finale_antwort
'''
mermaid(diagram, width=900)

In [ ]:
import gradio as gr


def stream_agent(message: str, history: list):
    """Generator: zeigt Recherchefortschritt und finale Antwort."""
    yield "Anfrage wird geprüft..."
    yield "Quellen werden gesucht..."
    antwort = chat_fn(message, history)
    yield antwort


stream_probe = list(stream_agent("Welchen Status hat Risiko R2?", []))
print(stream_probe[-1][:500])

demo_stream = gr.ChatInterface(
    fn=stream_agent,
    title="Meeting-Briefing-Agent Streaming",
    description="Recherchefortschritt und finale quellengebundene Antwort.",
    examples=["Welchen Status hat Risiko R2?", "Was wurde am 17.03.2026 entschieden?"],
    type="messages",
)
demo_stream.launch(quiet=True)


# 5 | Human-in-the-Loop UI
---


Das HITL-Muster bekommt hier eine Oberfläche für Freigabe. Der Meeting- & Research-Briefing-Agent bereitet eine Antwort mit Quelle und Sicherheit vor. Bei niedriger Sicherheit bleibt die finale Ausgabe gesperrt, bis eine Freigabe erfolgt.

```
Frage → Antwortentwurf + Quelle + Sicherheit
          ├─ Sicherheit hoch/mittel → anzeigen
          └─ Sicherheit niedrig     → Freigabe erforderlich
```

In [26]:
from langgraph.checkpoint.memory import InMemorySaver

# Checkpointer bleibt sichtbar, weil UI-Sessions und Freigaben Zustand brauchen.
hitl_checkpointer = InMemorySaver()
print("✅ HITL-Session-Checkpointer bereit")


✅ HITL-Session-Checkpointer bereit


In [27]:
# --- VIZ ---
hitl_diagram = """
%%{init: {'theme':'forest'}}%%
flowchart LR
    Q["Frage"] --> D["Antwortentwurf"]
    D --> S{"Sicherheit niedrig?"}
    S -->|nein| A["Antwort anzeigen"]
    S -->|ja| H["Freigabe erforderlich"]
    H -->|Genehmigen| A
    H -->|Ablehnen| R["Nicht freigegeben"]
"""
mermaid(hitl_diagram, width=800)


In [ ]:
import gradio as gr

sitzung = {"ausstehend": None}


def erstelle_entwurf(frage: str) -> dict:
    """Erstellt deterministisch einen Briefing-Entwurf mit Quelle und Sicherheit."""
    snippet = _waehle_snippet(frage)
    return {
        "frage": frage,
        "antwort": f"Antwort: {snippet['passage']}",
        "quelle": snippet["quelle"],
        "sicherheit": snippet["sicherheit"],
    }


def sende_anfrage(nachricht: str, verlauf: list):
    """Erstellt Entwurf und fordert bei niedriger Sicherheit Freigabe an."""
    verlauf = verlauf or []
    verlauf.append({"role": "user", "content": nachricht})
    entwurf = erstelle_entwurf(nachricht)
    text = (
        f"{entwurf['antwort']}\n\n"
        f"Quelle: {entwurf['quelle']}\n"
        f"Sicherheit: {entwurf['sicherheit']}"
    )
    if entwurf["sicherheit"] == "niedrig":
        sitzung["ausstehend"] = entwurf
        verlauf.append({"role": "assistant", "content": f"Freigabe erforderlich:\n\n{text}"})
        return "", verlauf, gr.update(visible=True)
    sitzung["ausstehend"] = None
    verlauf.append({"role": "assistant", "content": text})
    return "", verlauf, gr.update(visible=False)


def genehmige(verlauf: list):
    """Gibt einen ausstehenden Briefing-Entwurf frei."""
    pending = sitzung.get("ausstehend")
    if not pending:
        return verlauf, gr.update(visible=False)
    verlauf.append({"role": "assistant", "content": f"Freigegeben:\n\n{pending['antwort']}\nQuelle: {pending['quelle']}"})
    sitzung["ausstehend"] = None
    return verlauf, gr.update(visible=False)


def lehne_ab(verlauf: list):
    """Lehnt einen ausstehenden Briefing-Entwurf ab."""
    sitzung["ausstehend"] = None
    verlauf.append({"role": "assistant", "content": "Nicht freigegeben. Keine finale Ausgabe."})
    return verlauf, gr.update(visible=False)


hitl_probe = erstelle_entwurf("Frage außerhalb des Korpus")
print(hitl_probe)

with gr.Blocks(title="Briefing HITL", theme=gr.themes.Soft()) as demo_hitl:
    gr.Markdown("## Meeting-Briefing-Agent mit Freigabe")
    chatbot = gr.Chatbot(type="messages", label="Chat", height=350)
    with gr.Row():
        eingabe = gr.Textbox(placeholder="Frage an den Projekt-Korpus", label="Eingabe", scale=4)
        senden_btn = gr.Button("Senden", variant="primary", scale=1)
    with gr.Row(visible=False) as approval_row:
        approve_btn = gr.Button("Freigeben", variant="primary")
        reject_btn = gr.Button("Ablehnen", variant="stop")

    senden_btn.click(sende_anfrage, [eingabe, chatbot], [eingabe, chatbot, approval_row])
    eingabe.submit(sende_anfrage, [eingabe, chatbot], [eingabe, chatbot, approval_row])
    approve_btn.click(genehmige, [chatbot], [chatbot, approval_row])
    reject_btn.click(lehne_ab, [chatbot], [chatbot, approval_row])

demo_hitl.launch(quiet=True)


In [29]:
#@markdown   <p><font size="4" color='green'>  LangSmith Trace-Analyse</font> </br></p>

import time as _t; _t.sleep(2)
show_trace("M30-Gradio-UI", limit=3, show_steps=True)


## LangSmith Trace — `M30-Gradio-UI`

| Run | Status | Dauer | Child-Runs |
|-----|--------|-------|------------|
| `M30-BasicResearchChat` | ✅ success | 2.2s | 0 |
| `M30-BlocksResearchChat` | ✅ success | 4.9s | 0 |
| `M30-BasicResearchChat` | ✅ success | 2.1s | 0 |


### Steps — letzter Run: `M30-BasicResearchChat`

| # | Typ | Name | Status | Dauer |
|---|-----|------|--------|-------|
| 1 | `chain` | `model` | ✅ | 1.4s |
| 2 | `chain` | `tools` | ✅ | 0.0s |
| ↳ | `tool` | `suche_quellen` | ✅ | 0.0s |
| 3 | `chain` | `model` | ✅ | 0.9s |

# A | Aufgaben
---


<p><font color='darkblue' size="4">
📌 <b>Wichtig</b>
</font></p>

Die Aufgabenstellungen unten bieten Anregungen — eigene Herausforderungen sind ausdrücklich willkommen.

**Hinweis zur Lösungshilfe:**
> Generative KI darf im Kurs als Lernunterstützung genutzt werden, z. B. um Fehlermeldungen zu verstehen, Teilschritte zu klären oder Code-Varianten zu prüfen.


**Grundlagen**

Eine eigene Research-Chat-UI mit mindestens zwei Tools bauen: Quellen suchen und Quellenbindung prüfen.

**✅ Erledigt wenn:** Die Chat-Funktion eine Antwort mit Quelle oder `Nicht im Korpus` liefert und mindestens ein Tool-Aufruf im Log sichtbar wird.


**Aufbau**

Eine `gr.Blocks`-App mit Chat, Quellenpanel und Tool-Log bauen.

**✅ Erledigt wenn:** Chat-Verlauf und Quellen-/Tool-Log getrennt sichtbar sind und der Log eine Quelle oder einen Nicht-im-Korpus-Hinweis enthält.


**Vertiefung**

Streaming und Human-in-the-Loop kombinieren: Fortschritt anzeigen und bei niedriger Sicherheit eine manuelle Freigabe verlangen.

**✅ Erledigt wenn:** Ein Entwurf mit niedriger Sicherheit nicht direkt final ausgegeben wird, sondern eine sichtbare Freigabe verlangt.


<p><font color='darkblue' size="4">
 <b>Viz</b>
</font></p>

- [KI-Agent](https://editor.p5js.org/ralf.bendig.rb/full/u3Ee0jtFo)
- [LangGraph](https://editor.p5js.org/ralf.bendig.rb/full/EUzaFq4C4)


# B | Dokumente zum Weiterlesen
---

Ergänzende Artikel aus der Kurs-Dokumentation:

- [Meeting- & Research-Briefing-Agent im Betrieb](https://ralf-42.github.io/Agenten/08-deployment-betrieb/meeting-research-briefing-agent.html)
- [Tool Use & Function Calling](https://ralf-42.github.io/Agenten/04-agenten-implementierung/entwurf/tool-use-function-calling.html)
- [Einsteiger LangGraph](https://ralf-42.github.io/Agenten/05-frameworks/einsteiger-langgraph.html)
- [Human-in-the-Loop](https://ralf-42.github.io/Agenten/04-agenten-implementierung/ablauf-zustand/human-in-the-loop.html)
- [Aus Entwicklung ins Deployment](https://ralf-42.github.io/Agenten/08-deployment-betrieb/aus-entwicklung-ins-deployment.html)
